# Signer-honest sign recognition from video

Runs the video pipeline on **this MacBook Pro**: Apple M4 Pro (12 CPU cores: 8P+4E, 16 GPU cores), 24 GB unified memory, PyTorch on **Metal / MPS**. There is no CUDA and no Colab T4 on this machine.

Select the kernel that uses `~/.venvs/slr` (Python 3.12) before running. Landmark models are small; `frame_*` RGB backbones reuse the image zoo and share the same 24 GB with macOS, so they are the ones that may need a smaller batch.

This is `backend-v2`: everything the image backend did, plus isolated *sign* recognition
from a clip. The logic lives in `backend-v2/src/slr/video_experiment.py`, so these cells
stay short.

**Why video changes the argument.** On still images, protocol B had to *recover* capture
sessions, because no static ASL corpus records who signed each frame - and
`slr.signer_check` measures how badly that recovery fragments real signers (211 of 217 at
the usable threshold). PopSign ships a signer id for every one of its 21 Deaf signers, so
here the honest split is exact rather than approximated:

| | protocol | what it measures |
| --- | --- | --- |
| R | random clip split | what most video SLR papers report |
| S | signer-disjoint | signers dealt whole. Exact, not recovered |
| O | official benchmark | the corpus's own published signer-independent split |

`R - S` is the share of the published headline that was signer memorisation. Because S is
exact, that is a measurement, not an estimate.

**On ten-second uploads.** No isolated-sign corpus contains them: WLASL averages 2.4s,
AUTSL 1.8s, PopSign 1.4s. A long upload is cut into overlapping training-clip-length
windows, every window is scored, and the API returns the timeline alongside the best
window. See `slr.clips`.

## 1. Environment and code

In [ ]:
import platform, subprocess, torch

def _sysctl(key: str) -> str:
    return subprocess.check_output(["sysctl", "-n", key], text=True).strip()

print("os:", platform.platform())
print("chip:", _sysctl("machdep.cpu.brand_string"))
print("cpu cores:", _sysctl("hw.ncpu"))
print("memory_gb:", int(_sysctl("hw.memsize")) // 2**30)
print("torch", torch.__version__,
      "| cuda", torch.cuda.is_available(),
      "| mps", torch.backends.mps.is_available())
assert torch.backends.mps.is_available(), "this notebook expects Apple Silicon MPS"

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for p in [ROOT, *ROOT.parents]:
    if (p / "backend-v2" / "src" / "slr").is_dir():
        ROOT = p
        break
else:
    raise SystemExit("could not find backend-v2/src/slr; open the notebook from the repo")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "backend-v2" / "src"))

from slr import clips, landmarks, sources, video_data, video_model as VM, video_train, video_experiment
print("repo", ROOT)
print("device", video_train.device())
print("loaded slr from", Path(video_experiment.__file__).parent)

## 2. Kaggle credentials

PopSign is a Kaggle **competition** dataset, so besides the API token you must accept the
competition rules once at
[kaggle.com/competitions/asl-signs/rules](https://www.kaggle.com/competitions/asl-signs/rules).
There is no API for accepting rules; a 403 here is a human step, not a bug.

On this machine, credentials live in `~/.kaggle/kaggle.json` (chmod 600). No key is ever
typed into a cell.

In [ ]:
from pathlib import Path

kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print("[kaggle] credentials present at", kaggle_json)
else:
    raise SystemExit(
        "missing ~/.kaggle/kaggle.json. Create a legacy API key at "
        "https://www.kaggle.com/settings and copy it there (chmod 600)."
    )

## 3. Data

`sources.main(["list"])` now lists the video and landmark corpora alongside the image
ones - one registry, every corpus declared exactly once.

PopSign is ~100k clips of MediaPipe Holistic landmarks, 250 signs, 21 Deaf signers. No
video: the landmarks *are* the release, which is also why the background shortcut the
image backend spent eighteen backbones fighting simply does not exist here.

In [ ]:
sources.main(["list"])

In [ ]:
video_experiment.run("data", corpus="popsign_islr")

## 4. Protocol R - the leaky baseline

A random clip split puts every signer on both sides. `shared_signers` in the result
counts how many signers the model trained on and is then tested on; for a random split
that is all of them.

`PER_CLASS` caps clips per sign by dealing whole signers, so subsampling cannot itself
put one signer on both sides.

In [ ]:
PER_CLASS = 0          # clips per sign; 0 uses all ~100k
PRESET    = "lm_transformer"
EPOCHS    = 10

video_experiment.run("R", per_class=PER_CLASS, preset=PRESET, epochs=EPOCHS)

## 5. Protocol S - signer-disjoint

Same corpus, same backbone, same epochs. The only change is that signers are dealt whole.
The stage asserts `shared_signers == 0` rather than trusting the split code.

In [ ]:
video_experiment.run("S", per_class=PER_CLASS, preset=PRESET, epochs=EPOCHS)

## 6. The headline table

In [ ]:
video_experiment.summary()

## 7. Temporal architecture search

Ranked on **validation** macro-F1 under the signer-disjoint split; only the winner is run
against test. Three questions the zoo is built to answer:

- **Does motion matter at all?** `lm_mean` averages per-frame features and is order-blind
  by construction - a clip played backwards scores identically. If it ties the sequence
  models, these signs are separable from a single handshape.
- **Attention or just recurrence?** `lm_gru` against `lm_transformer` at matched width.
  Sixteen frames is a short sequence.
- **Landmarks or pixels?** `lm_*` against `frame_*`, the latter reusing the image zoo one
  backbone per frame. If pixels win, the background shortcut was worth more than geometry.

The sweep is resumable: the board is rewritten after every candidate.

In [ ]:
print({k: v for k, v in VM.SWEEPS.items()})

In [ ]:
SEARCH = "landmark"    # "quick" | "landmark" | "rgb" | "all"

arch = video_experiment.run("arch", per_class=PER_CLASS,
                            presets=VM.SWEEPS[SEARCH], epochs=EPOCHS)
BEST = arch["winner"]
print("winner:", BEST, "| test:", arch["winner_test"])

In [ ]:
import pandas as pd

board = pd.DataFrame(arch["leaderboard"])
board["f1_per_Mparam"] = (board.val_f1 / board.params_m * 100).round(3)
board

## 8. Calibration and abstention

Temperature fitted on validation only. Abstention matters more here than on stills: a
long upload is cut into windows and most windows hold no complete sign, so a model that
cannot say "nothing here" will label the gaps confidently.

In [ ]:
cal = video_experiment.run("cal", per_class=PER_CLASS, winner=BEST)
cal["test"]["risk_coverage"]

## 9. End to end on a clip

Windows a clip the way the API does and prints the timeline, so you can see what was
actually classified and when rather than one label for the whole upload.

Needs `mediapipe` and `opencv`, which are only required for pixels-to-landmarks - not for
training on PopSign, and not for the test suite. Set `CLIP` to a local `.mp4`.

In [ ]:
import numpy as np, torch
from pathlib import Path
from slr import video_train

CLIP = Path("sign.mp4")  # local clip on this Mac
assert CLIP.exists(), f"set CLIP to a real video path; {CLIP} is missing"

dev = video_train.device()
blob = torch.load(f"runs/S_arch_WINNER_{BEST}/best.pt", map_location="cpu", weights_only=False)
net = VM.build(blob["preset"], len(blob["classes"])).to(dev).eval()
net.load_state_dict(blob["state_dict"])

total, fps = clips.probe(CLIP)
print(clips.duration_warning(total / fps) or "clip is about one sign long")
print("scoring on", dev)

wins = clips.windows(CLIP, n_frames=blob["n_frames"])
x = torch.from_numpy(np.stack([landmarks.to_features(landmarks.extract(w.frames))
                               for w in wins])).float().to(dev)
with torch.no_grad():
    probs = torch.softmax(net(x), dim=1).cpu()

for w, p in zip(wins, probs):
    k = int(p.argmax())
    print(f"{w.start:5.1f}-{w.end:4.1f}s  {blob['classes'][k]:<20} p={float(p[k]):.3f}")